# Filter Comparison — FIR, IIR, and Kalman

All filter implementations from this project applied to SPY with:

- **Causal-only** implementations (no `filtfilt`) — fair lag comparison
- A consistent span/order of ~50 trading days where applicable
- Quantitative metrics from `metrics.py`: RMSE, MAE, noise reduction, lag

| Filter | Type | Key property |
|--------|------|-------------|
| SMA 50 | FIR  | Flat frequency response (boxcar) |
| WMA 50 | FIR  | Linearly weighted, less Gibbs ripple |
| EMA 50 | IIR  | Infinite memory, exponential weights |
| Butterworth LP | IIR | Maximally flat passband, fast rolloff |
| Kalman 1D | Adaptive IIR | Optimal under Gaussian noise, no fixed lag |
| Kalman 2D | Adaptive IIR | Also estimates trend velocity |


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import signal as sp_signal

from kalman_filter import KalmanFilter1D, KalmanFilter2D
import metrics
from filters import load_asset, DATA_DIR

plt.rcParams['figure.dpi'] = 110

df    = load_asset('SPY', DATA_DIR)
price = df['Close']
meas  = price.values
idx   = price.index
print(f'SPY: {len(price)} observations  ({idx[0].date()} → {idx[-1].date()})')


In [ ]:
# ── Apply all 6 filters causally ───────────────────────────────────────────
M    = 50         # common window span in trading days
dvar = float(np.var(np.diff(meas)))

# FIR: SMA 50 — causal rolling mean
sma50 = pd.Series(meas).rolling(M).mean().values

# FIR: WMA 50 — linearly decaying weights
weights = np.arange(1, M + 1, dtype=float)
wma50   = (pd.Series(meas)
            .rolling(M)
            .apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
            .values)

# IIR: EMA 50 — exponential moving average
ema50 = pd.Series(meas).ewm(span=M, adjust=False).mean().values

# IIR: Butterworth 4th-order low-pass (causal via sosfilt)
Wn    = (1.0 / M) / 0.5
sos   = sp_signal.butter(4, Wn, btype='low', output='sos')
butter50 = sp_signal.sosfilt(sos, meas)

# Kalman 1D
kf1 = KalmanFilter1D(Q=dvar * 0.01, R=dvar * 2.0, x0=meas[0])
kalman1d, _, _ = kf1.filter(meas)

# Kalman 2D (price estimate only for the comparison)
kf2 = KalmanFilter2D(
    Q=np.diag([dvar * 0.5, dvar * 1e-4]),
    R=dvar * 100.0,
    x0=np.array([meas[0], 0.0]),
)
kalman2d, _, _ = kf2.filter(meas)

filters_dict = {
    'SMA 50':          sma50,
    'WMA 50':          wma50,
    'EMA 50':          ema50,
    'Butterworth LP':  butter50,
    'Kalman 1D':       kalman1d,
    'Kalman 2D':       kalman2d,
}
print('All 6 filters computed.')


In [ ]:
# ── Reference 'truth' signal ───────────────────────────────────────────────
# We use a very long zero-phase Butterworth to approximate the underlying trend.
# This is the best we can do without knowing the true price process.
b_ref, a_ref = sp_signal.butter(4, (1.0/300)/0.5, btype='low')
reference    = sp_signal.filtfilt(b_ref, a_ref, meas)

rows = [metrics.summarize('Raw price (no filter)', reference, meas)]
for name, est in filters_dict.items():
    valid = np.isfinite(est)
    row   = metrics.summarize(name, reference[valid], est[valid])
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('Filter')
print('Metrics table:')
comparison_df


In [ ]:
# ── Overlay plot — last 500 trading days ───────────────────────────────────
WINDOW = -500
plot_colors = ['#9E9E9E', '#FF9800', '#4CAF50', '#9C27B0', '#FF5722',
               '#2196F3', '#795548']

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(idx[WINDOW:], meas[WINDOW:],
        color='#BDBDBD', lw=0.8, alpha=0.6, label='SPY raw', zorder=2)

for (name, est), color in zip(filters_dict.items(), plot_colors):
    ax.plot(idx[WINDOW:], est[WINDOW:], lw=1.4, label=name, color=color, zorder=3)

ax.set_title('SPY — All Filters (last 500 trading days, causal only)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=9, ncol=2, framealpha=0.92)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Kalman gain over time — shows how adaptive IIR differs from fixed-weight FIR
fig, ax = plt.subplots(figsize=(13, 3))
kf1_gain = KalmanFilter1D(Q=dvar * 0.01, R=dvar * 2.0, x0=meas[0])
_, gains, variances = kf1_gain.filter(meas)

ax.plot(idx, gains, color='#FF5722', lw=0.8, label='Kalman gain  K')
ax.set_title('Kalman 1D — Gain Convergence', fontsize=11)
ax.set_ylabel('Kalman Gain')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Steady-state gain: {gains[-1]:.6f}  variance: {variances[-1]:.6f}')
